In [1]:
import napari
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from scipy.spatial import ConvexHull, Delaunay

In [2]:
folder_no_organoid = Path(r"Z:\Bel\Individual_Vascumap_Outputs\Defne_Vascumap\With_Interim\270325.2_Device 2_Merged_img0")
folder_organoid = Path(r"Z:\Bel\Individual_Vascumap_Outputs\Defne_Vascumap\With_Interim\marina_2024.11.14_MCL1_MCL3_M4_morphologyBFseeding1_Bead_dev1_Merged_img0")

In [3]:
# Load every output file produced for one image into a dict, keyed by suffix.
# This is more robust than positional indexing because the file order from
# `iterdir()` depends on filesystem order.

VOXEL_SIZE_UM = np.array([2.0, 2.0, 2.0])  # default VascuMap voxel size

SUFFIX_MAP = {
    'original':       '_cropped_stack_aligned.npy',
    'translation':    '_vessel_translation_aligned.npy',
    'segmentation':   '_clean_segmentation.npy',
    'skeleton':       '_skeleton.npy',
    'organoid_mask':  '_organoid_mask.npy',          # only present if an organoid was masked
    'holes':          '_holes.npy',                   # only present if internal pores were detected
    'clean_graph':    '_clean_graph.pkl',             # only present in newer outputs
    'branch_metrics': '_branch_metrics.csv',
}

def load_dataset(folder: Path) -> dict:
    """Load every known VascuMap output file from `folder` into a dict.

    Missing files are silently skipped so the same loader works on both
    organoid and no-organoid runs.
    """
    folder = Path(folder)
    out = {'folder': folder, 'name': folder.name}
    for key, suffix in SUFFIX_MAP.items():
        matches = list(folder.glob(f'*{suffix}'))
        if not matches:
            continue
        path = matches[0]
        if path.suffix == '.npy':
            out[key] = np.load(path)
        elif path.suffix == '.csv':
            out[key] = pd.read_csv(path)
        elif path.suffix == '.pkl':
            with open(path, 'rb') as f:
                out[key] = pickle.load(f)
    return out

In [4]:
ds_no_org = load_dataset(folder_no_organoid)
ds_org    = load_dataset(folder_organoid)

print('No-organoid keys :', sorted(k for k in ds_no_org if k not in {'folder', 'name'}))
print('Organoid keys    :', sorted(k for k in ds_org    if k not in {'folder', 'name'}))
print('seg shape (no org):', ds_no_org['segmentation'].shape)
print('seg shape (org)   :', ds_org['segmentation'].shape)

No-organoid keys : ['branch_metrics', 'original', 'segmentation', 'skeleton', 'translation']
Organoid keys    : ['branch_metrics', 'clean_graph', 'holes', 'organoid_mask', 'original', 'segmentation', 'skeleton', 'translation']
seg shape (no org): (44, 3283, 1836)
seg shape (org)   : (68, 2909, 2783)


In [5]:
# Hull-volume helpers.  These mirror the convention used in
# `vascumap/skeletonisation.py::clean_and_analyse`: take the convex hull of
# every vessel-positive voxel (in micrometres) and rasterise its interior
# back onto the image grid using `Delaunay.find_simplex >= 0`.

def compute_hull_mask(segmentation: np.ndarray,
                      voxel_size_um=VOXEL_SIZE_UM) -> np.ndarray:
    """Return a 3D bool mask of the convex hull of `segmentation > 0`.

    The mask has the same shape as `segmentation`.  Hull membership is
    evaluated only inside the hull's bounding box for speed.
    """
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)
    seg_pts = np.argwhere(segmentation > 0)
    if seg_pts.shape[0] < 4:
        return np.zeros_like(segmentation, dtype=bool)

    hull = ConvexHull(seg_pts * voxel_size_um)
    delaunay = Delaunay(hull.points[hull.vertices])

    # Only test voxels inside the hull's bounding box.
    lo = seg_pts.min(axis=0)
    hi = seg_pts.max(axis=0) + 1
    zz, yy, xx = np.meshgrid(
        np.arange(lo[0], hi[0]),
        np.arange(lo[1], hi[1]),
        np.arange(lo[2], hi[2]),
        indexing='ij',
    )
    box_idx = np.stack([zz.ravel(), yy.ravel(), xx.ravel()], axis=1)
    inside = delaunay.find_simplex(box_idx * voxel_size_um) >= 0

    mask = np.zeros_like(segmentation, dtype=bool)
    mask[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]] = inside.reshape(zz.shape)
    return mask


def subtract_organoid_from_hull(hull_mask: np.ndarray,
                                organoid_mask_xy: np.ndarray) -> np.ndarray:
    """Return `hull_mask` minus the z-extruded organoid footprint.

    `organoid_mask_xy` is a 2D (Y, X) bool mask.  It is broadcast over every
    z-slice of the hull (matching how the analysis pipeline excludes the
    organoid volume from V_hull).
    """
    organoid_mask_xy = organoid_mask_xy.astype(bool)
    if organoid_mask_xy.shape != hull_mask.shape[1:]:
        raise ValueError(
            f'organoid mask {organoid_mask_xy.shape} does not match hull '
            f'XY shape {hull_mask.shape[1:]}'
        )
    extruded = np.broadcast_to(organoid_mask_xy[None, :, :], hull_mask.shape)
    return hull_mask & ~extruded


def voxel_volume_um3(mask: np.ndarray, voxel_size_um=VOXEL_SIZE_UM) -> float:
    return float(mask.sum()) * float(np.prod(voxel_size_um))

In [6]:
# Compute hull masks for both datasets.  This is the slow step (a few
# seconds per dataset) so we do it once and reuse the masks below.

hull_no_org = compute_hull_mask(ds_no_org['segmentation'])
hull_org    = compute_hull_mask(ds_org['segmentation'])
hull_org_corrected = subtract_organoid_from_hull(hull_org, ds_org['organoid_mask'])

print(f"V_hull (no organoid)        : {voxel_volume_um3(hull_no_org):,.0f} um^3")
print(f"V_hull (organoid, raw)      : {voxel_volume_um3(hull_org):,.0f} um^3")
print(f"V_hull (organoid, corrected): {voxel_volume_um3(hull_org_corrected):,.0f} um^3")
print(f"  -> organoid bite          : "
      f"{voxel_volume_um3(hull_org) - voxel_volume_um3(hull_org_corrected):,.0f} um^3")

V_hull (no organoid)        : 2,082,590,560 um^3
V_hull (organoid, raw)      : 4,390,863,352 um^3
V_hull (organoid, corrected): 3,946,625,880 um^3
  -> organoid bite          : 444,237,472 um^3


## View 1 — no-organoid dataset

Layers, top to bottom:

* **original**  — raw cropped stack
* **segmentation** — `_clean_segmentation.npy` rendered as labels
* **skeleton** — single-voxel-thick centreline overlaid in cyan
* **hull volume** — translucent yellow box of the convex hull used as `V_hull`
  in every `*_per_volume` metric

In [7]:
viewer1 = napari.Viewer(title=f'no organoid — {ds_no_org["name"]}')
viewer1.add_image(ds_no_org['original'], name='original',
                  scale=VOXEL_SIZE_UM, contrast_limits=None)
viewer1.add_labels(ds_no_org['segmentation'].astype(np.uint8),
                   name='segmentation', scale=VOXEL_SIZE_UM, opacity=0.5)
viewer1.add_image(hull_no_org.astype(np.float32), name='hull volume',
                  scale=VOXEL_SIZE_UM, colormap='yellow',
                  blending='additive', opacity=0.15)
viewer1.add_image(ds_no_org['skeleton'].astype(np.float32), name='skeleton',
                  scale=VOXEL_SIZE_UM, colormap='cyan',
                  blending='additive')
viewer1.dims.ndisplay = 3

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## View 2 — organoid dataset, with hull correction

Layers, top to bottom:

* **original** — raw cropped stack
* **segmentation** — vessel labels
* **organoid mask** (red) — the 2D footprint excluded from `V_hull`,
  z-extruded across the whole stack
* **hull (raw)** (yellow) — convex hull of the segmentation, *before* the
  organoid is subtracted
* **hull (organoid-corrected)** (green) — what the analysis actually uses
  as `V_hull`.  The bite taken out of the yellow box is the organoid
  contribution to all `*_per_volume` densities.
* **skeleton** (cyan)

In [8]:
organoid_extruded = np.broadcast_to(
    ds_org['organoid_mask'].astype(bool)[None, :, :],
    ds_org['segmentation'].shape,
).copy()

viewer2 = napari.Viewer(title=f'organoid — {ds_org["name"]}')
viewer2.add_image(ds_org['original'], name='original',
                  scale=VOXEL_SIZE_UM)
viewer2.add_labels(ds_org['segmentation'].astype(np.uint8),
                   name='segmentation', scale=VOXEL_SIZE_UM, opacity=0.5)
viewer2.add_image(organoid_extruded.astype(np.float32), name='organoid mask',
                  scale=VOXEL_SIZE_UM, colormap='red',
                  blending='additive', opacity=0.25)
viewer2.add_image(hull_org.astype(np.float32), name='hull (raw)',
                  scale=VOXEL_SIZE_UM, colormap='yellow',
                  blending='additive', opacity=0.10, visible=False)
viewer2.add_image(hull_org_corrected.astype(np.float32),
                  name='hull (organoid-corrected)',
                  scale=VOXEL_SIZE_UM, colormap='green',
                  blending='additive', opacity=0.15)
viewer2.add_image(ds_org['skeleton'].astype(np.float32), name='skeleton',
                  scale=VOXEL_SIZE_UM, colormap='cyan',
                  blending='additive')
viewer2.dims.ndisplay = 3

## View 3 — graph-derived parameters (organoid dataset)

Using `_branch_metrics.csv`, overlay one vector per branch on the organoid
viewer:

* **branches** (white) — every cleaned branch with `is_sprout == False`
* **sprouts**  (magenta) — dead-end branches counted in `n_sprouts`
* **junctions** (yellow points) — every node that is an endpoint of two or
  more branches; this is what `n_junctions` / `junctions_per_volume` count.

This makes the difference between *branch_only* metrics (which collapse
A—J—B chains where J only carries a sprout) and the full cleaned graph
visible at a glance.

In [ ]:
def _pick(df, *candidates):
    """Return the first column name from `candidates` that is in `df`."""
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f'none of {candidates} found in branch_metrics columns: '
                   f'{list(df.columns)}')

if 'branch_metrics' in ds_org:
    bm = ds_org['branch_metrics']

    sz = _pick(bm, 'start_z_um', 'start_z', 'node_a_z_um', 'node_a_z')
    sy = _pick(bm, 'start_y_um', 'start_y', 'node_a_y_um', 'node_a_y')
    sx = _pick(bm, 'start_x_um', 'start_x', 'node_a_x_um', 'node_a_x')
    ez = _pick(bm, 'end_z_um',   'end_z',   'node_b_z_um', 'node_b_z')
    ey = _pick(bm, 'end_y_um',   'end_y',   'node_b_y_um', 'node_b_y')
    ex = _pick(bm, 'end_x_um',   'end_x',   'node_b_x_um', 'node_b_x')
    sprout_col = _pick(bm, 'is_sprout')

    starts_um = bm[[sz, sy, sx]].to_numpy(dtype=float)
    ends_um   = bm[[ez, ey, ex]].to_numpy(dtype=float)

    # If the picked columns are the voxel-index variants, scale to µm so the
    # vectors line up with the image layers (which are also in µm via `scale`).
    if not sz.endswith('_um'):
        starts_um = starts_um * VOXEL_SIZE_UM
        ends_um   = ends_um   * VOXEL_SIZE_UM

    is_sprout = bm[sprout_col].to_numpy(dtype=bool)
    vectors = np.stack([starts_um, ends_um - starts_um], axis=1)  # (N, 2, 3)

    if (~is_sprout).any():
        viewer2.add_vectors(vectors[~is_sprout], name='branches',
                            edge_color='white', edge_width=2.0,
                            length=1.0, vector_style='line')
    if is_sprout.any():
        viewer2.add_vectors(vectors[is_sprout], name='sprouts',
                            edge_color='magenta', edge_width=2.0,
                            length=1.0, vector_style='line')

    # Junctions = endpoints shared by 2+ non-sprout branches.
    branch_endpoints = np.vstack([
        starts_um[~is_sprout], ends_um[~is_sprout],
    ])
    if branch_endpoints.size:
        rounded = np.round(branch_endpoints, 3)
        unique_pts, counts = np.unique(rounded, axis=0, return_counts=True)
        junction_pts = unique_pts[counts >= 2]
        if junction_pts.size:
            viewer2.add_points(junction_pts, name='junctions',
                               face_color='yellow', size=8,
                               symbol='disc', opacity=0.9)

    print(f'branches : {(~is_sprout).sum()}')
    print(f'sprouts  : { is_sprout.sum()}')
else:
    print('No _branch_metrics.csv in the organoid folder; skipping graph view.')